# Fine-tune Llama on Bhagavad Gita with MLX (Optimized for M4 24GB)

This notebook fine-tunes a Llama model using Apple's [MLX](https://github.com/ml-explore/mlx) framework, optimized for the **MacBook Air M4 with 24GB unified memory**.

## Model Options
| Model | Peak RAM | Quality | Training Time (~1000 iters) |
|---|---|---|---|
| `Llama-3.2-3B-Instruct-4bit` | ~10 GB | Good | ~45 min |
| `Meta-Llama-3.1-8B-Instruct-4bit` | ~16 GB | **Better** (recommended) | ~90 min |

## Key Improvements Over Baseline
- **Full-layer LoRA**: All transformer layers trained (not just 16)
- **Masked prompts**: Loss computed only on answers, not questions
- **Cosine LR schedule with warmup**: Stable convergence
- **LoRA alpha = rank**: Prevents scaling instability (was alpha=160 before)
- **Gradient accumulation**: Simulates larger effective batch size
- **1000+ iterations**: ~1 full epoch (was 20 iters = <2% of one epoch)

## 1. Install Dependencies

In [ ]:
%uv pip install mlx-lm

## 2. Prepare Dataset
MLX prefers datasets in a specific format (usually training and validation files in a directory). We will take our existing `gita_qna_for_finetune.jsonl` and split it into `train.jsonl` and `valid.jsonl` in a `data/mlx` directory.

In [ ]:
import json
import os
import random

# Config
SOURCE_FILE = "data/processed/gita_qna_for_finetune.jsonl"
MLX_DATA_DIR = "data/mlx"
TRAIN_FILE = os.path.join(MLX_DATA_DIR, "train.jsonl")
VALID_FILE = os.path.join(MLX_DATA_DIR, "valid.jsonl")
SPLIT_RATIO = 0.9  # 90% train, 10% validation

os.makedirs(MLX_DATA_DIR, exist_ok=True)

if not os.path.exists(SOURCE_FILE):
    print(f"ERROR: Source file {SOURCE_FILE} not found.")
else:
    with open(SOURCE_FILE, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f]

    print(f"Loaded {len(data)} total examples.")

    # --- Deduplicate by serialised message content ---
    seen = set()
    deduped = []
    for item in data:
        key = json.dumps(item["messages"], ensure_ascii=False, sort_keys=True)
        if key not in seen:
            seen.add(key)
            deduped.append(item)

    removed = len(data) - len(deduped)
    print(f"Removed {removed} duplicates → {len(deduped)} unique examples remain.")
    data = deduped

    # --- Shuffle and split ---
    random.seed(42)
    random.shuffle(data)

    split_idx = int(len(data) * SPLIT_RATIO)
    train_data = data[:split_idx]
    valid_data = data[split_idx:]

    print(f"Saving {len(train_data)} train examples to {TRAIN_FILE}...")
    with open(TRAIN_FILE, "w", encoding="utf-8") as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"Saving {len(valid_data)} valid examples to {VALID_FILE}...")
    with open(VALID_FILE, "w", encoding="utf-8") as f:
        for item in valid_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print("Done!")

## 3. Configuration

Choose your model based on your priority. For **M4 24GB**, the 8B model is recommended — it fits
comfortably with ~16 GB peak and produces noticeably better philosophical reasoning.

Set `USE_8B = False` to switch to the faster 3B option.

In [ ]:
USE_8B = True  # Set False to use 3B (faster, less RAM, ~45 min training)

if USE_8B:
    # Llama 3.1 8B — best quality, ~16 GB peak on 24GB M4
    MODEL_NAME = "mlx-community/Meta-Llama-3.1-8B-Instruct-4bit"
    FUSED_SAVE_PATH = "models/gita-llama-3.1-8b-fused"
    # Training config for 8B
    ITERS = 1000
    BATCH_SIZE = 4
    GRAD_ACCUM = 4          # Effective batch = 16
    NUM_LAYERS = 32         # All 32 layers
    LORA_RANK = 16
    LORA_ALPHA = 16         # alpha == rank → scale = 1.0 (stable)
    LR = "2e-5"
else:
    # Llama 3.2 3B — faster training, ~10 GB peak
    MODEL_NAME = "mlx-community/Llama-3.2-3B-Instruct-4bit"
    FUSED_SAVE_PATH = "models/gita-llama-3.2-3b-fused"
    # Training config for 3B
    ITERS = 1200
    BATCH_SIZE = 8
    GRAD_ACCUM = 2          # Effective batch = 16
    NUM_LAYERS = 28         # All 28 layers
    LORA_RANK = 32
    LORA_ALPHA = 32
    LR = "1e-4"

print(f"Model : {MODEL_NAME}")
print(f"Iters : {ITERS} | Batch: {BATCH_SIZE} | Grad accum: {GRAD_ACCUM} | Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Layers: {NUM_LAYERS} | LoRA rank: {LORA_RANK} | alpha: {LORA_ALPHA} | LR: {LR}")

## 4. Run Fine-Tuning

Training is done via `mlx_lm.lora`. Output is also saved to `training.log` so we can
plot the loss curve in the next cell.

### Key parameter decisions
| Parameter | Value | Why |
|---|---|---|
| `--iters` | 1000–1200 | ~1 full epoch (was 20 = <2% of 1 epoch) |
| `--num-layers` | all (28/32) | Full model adaptation; 24GB has headroom |
| `--lora-rank` / `--lora-alpha` | equal | Keeps LoRA scale=1.0, prevents instability |
| `--lora-dropout` | 0.05 | Light regularization on 4.6K examples |
| `--mask-prompt` | on | Loss only on assistant answers, not questions |
| `--lr-schedule cosine_decay` | + warmup 100 | Smooth convergence, avoids late overfitting |
| `--grad-accumulation-steps` | 2–4 | Effective batch = 16 without extra memory |

In [ ]:
import yaml

# mlx_lm.lora does NOT accept --lora-rank / --lora-alpha / --lora-dropout /
# --lr-schedule / --warmup as CLI flags.
# These must be passed via a YAML config file using the -c flag.
# scale = lora_alpha / lora_rank  →  setting both equal gives scale = 1.0 (stable)

lora_train_config = {
    # --- Model & data ---
    "model":      MODEL_NAME,
    "data":       MLX_DATA_DIR,
    "adapter_path": "adapters",
    # --- Training loop ---
    "train":      True,
    "iters":      ITERS,
    "batch_size": BATCH_SIZE,
    "grad_accumulation_steps": GRAD_ACCUM,
    "num_layers": NUM_LAYERS,
    "mask_prompt": True,
    "max_seq_length": 2048,
    # --- Eval & checkpointing ---
    "steps_per_report": 10,
    "steps_per_eval":   100,
    "val_batches":      50,
    "save_every":       200,
    # --- Learning rate ---
    "learning_rate": float(LR),
    "lr_schedule": {
        "name": "cosine_decay",
        "warmup": 100,
        "arguments": [float(LR), ITERS],   # [init_lr, decay_steps]
    },
    # --- LoRA parameters (NOT available as CLI flags) ---
    "lora_parameters": {
        "rank":    LORA_RANK,
        "scale":   1.0,        # = lora_alpha / lora_rank; 1.0 is the stable default
        "dropout": 0.05,
    },
}

CONFIG_PATH = "lora_train_config.yaml"
with open(CONFIG_PATH, "w") as f:
    yaml.dump(lora_train_config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to {CONFIG_PATH}")
print("-" * 40)
print(open(CONFIG_PATH).read())

In [ ]:
# All parameters are in lora_train_config.yaml — nothing to override here.
# The -c flag is the only argument needed.
!mlx_lm.lora -c {CONFIG_PATH} 2>&1 | tee training.log

## 5. Test Inference

Load the fine-tuned model with adapters and run a quick sanity check.
We use a helper function so we can reuse it in the evaluation section below.

In [ ]:
from mlx_lm import load, generate

# Load fine-tuned model with LoRA adapters
model, tokenizer = load(MODEL_NAME, adapter_path="adapters")

SYSTEM_PROMPT = "You are a wise teacher drawing from the Bhagavad Gita. Offer grounded, practical wisdom in plain modern language. Do not quote verses directly — speak as a teacher would."

def ask(query: str, max_tokens: int = 300) -> str:
    """Generate an answer using the fine-tuned model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": query},
    ]
    # Use the tokenizer's chat template if available, otherwise fall back to manual format
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n"
            f"{query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"
        )
    response = generate(model, tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)
    return response

# Quick sanity check
query = "What is the nature of duty?"
print(f"Q: {query}\n")
print(ask(query))

## 6. Fuse Adapters into Base Model

Merges the LoRA adapter weights into the base model weights, producing a single
self-contained model file ready for deployment with `mlx-lm` or conversion to GGUF.

The fused model is saved to the path defined in the configuration cell (`FUSED_SAVE_PATH`).

In [ ]:
!mlx_lm.fuse \
    --model {MODEL_NAME} \
    --adapter-path "adapters" \
    --save-path {FUSED_SAVE_PATH}

print(f"\nFused model saved to: {FUSED_SAVE_PATH}")

# Verify the output files exist
import os
if os.path.exists(FUSED_SAVE_PATH):
    files = os.listdir(FUSED_SAVE_PATH)
    total_size = sum(
        os.path.getsize(os.path.join(FUSED_SAVE_PATH, f))
        for f in files
    ) / (1024 ** 3)
    print(f"Files : {files}")
    print(f"Total size: {total_size:.2f} GB")

In [ ]:
from mlx_lm import load, generate

# Load model and tokenizer with adapters
model, tokenizer = load(MODEL_NAME, adapter_path="adapters")

# Define a prompt template (Standard Llama 3)
prompt_template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a wise teacher drawing from Bhagavad Gita.<|eot_id|><|start_header_id|>user<|end_header_id|>

{}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

query = "Using Chapter 2, Verse 47, answer: What is the nature of duty?"
prompt = prompt_template.format(query)

response = generate(model, tokenizer, prompt=prompt, verbose=True)

In [ ]:
!mlx_lm.fuse \
    --model {MODEL_NAME} \
    --adapter-path "adapters" \
    --save-path "models/gita-llama-3.2-3b-fused"